In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
from pathlib import Path

split_dir = Path("/content/drive/MyDrive/NLP/splits")

train_df = pd.read_csv(split_dir / "train.csv", keep_default_na=False)
val_df = pd.read_csv(split_dir / "validation.csv", keep_default_na=False)
test_df = pd.read_csv(split_dir / "test.csv", keep_default_na=False)

X_train = train_df["review_comment"].to_numpy(dtype=str)
X_val = val_df["review_comment"].to_numpy(dtype=str)
X_test = test_df["review_comment"].to_numpy(dtype=str)

y_train = train_df["label"].to_numpy(dtype="int32")
y_val = val_df["label"].to_numpy(dtype="int32")
y_test = test_df["label"].to_numpy(dtype="int32")

In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models

tf.keras.utils.set_random_seed(42)

BATCH_SIZE = 64
MAX_TOKENS = 15000

def make_dataset(texts, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((texts, labels))

    if training:
        ds = ds.shuffle(
            buffer_size=len(texts),
            seed=42
        )

    return ds.batch(BATCH_SIZE).prefetch(1)

In [4]:
train_ds = make_dataset(X_train, y_train, training=True)
val_ds = make_dataset(X_val, y_val)
test_ds = make_dataset(X_test, y_test)

In [5]:
vectorizer = layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="tf_idf"
)

adapt_ds = tf.data.Dataset.from_tensor_slices(X_train).batch(BATCH_SIZE)
vectorizer.adapt(adapt_ds)

In [6]:
model = models.Sequential([
    layers.Input(shape=(), dtype=tf.string),
    vectorizer,
    layers.Dense(5, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, 15000)          │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │        75,005 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 75,005 (292.99 KB)

 Trainable params: 75,005 (292.99 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True
        )
    ]
)

Epoch 1/50
407/407 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.6614 - loss: 1.0736 - val_accuracy: 0.6909 - val_loss: 0.9306
Epoch 2/50
407/407 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.7447 - loss: 0.7633 - val_accuracy: 0.6906 - val_loss: 0.8974
Epoch 3/50
407/407 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.7833 - loss: 0.6535 - val_accuracy: 0.6847 - val_loss: 0.9051
Epoch 4/50
407/407 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.8065 - loss: 0.5896 - val_accuracy: 0.6836 - val_loss: 0.9223
Epoch 5/50
407/407 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.8199 - loss: 0.5464 - val_accuracy: 0.6790 - val_loss: 0.9468


In [8]:
import numpy as np
from sklearn.metrics import classification_report, mean_absolute_error

# Validation predictions while developing the model
y_pred = np.argmax(model.predict(val_ds), axis=1)

print(classification_report(
    y_val,
    y_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=["Score 1", "Score 2", "Score 3", "Score 4", "Score 5"],
    zero_division=0
))

print("MAE:", mean_absolute_error(y_val, y_pred))

102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
              precision    recall  f1-score   support

     Score 1       0.68      0.86      0.76      1383
     Score 2       0.15      0.08      0.10       339
     Score 3       0.27      0.11      0.16       564
     Score 4       0.38      0.13      0.19       950
     Score 5       0.77      0.94      0.85      3266

    accuracy                           0.69      6502
   macro avg       0.45      0.42      0.41      6502
weighted avg       0.62      0.69      0.63      6502

MAE: 0.48815749000307596


In [9]:
# from pathlib import Path

# model_dir = Path("/content/drive/MyDrive/NLP/models")
# model_dir.mkdir(parents=True, exist_ok=True)

# model_path = model_dir / "review_score_model.keras"

# model.save(str(model_path))

# print("Model saved:", model_path)